# Fit all models to antigen-prime data

## Import packages

In [1]:
import pickle
import glob as glob
import pandas as pd

# Numerical libraries
import jax.numpy as jnp
import evofr as ef
from numpyro.infer.autoguide import AutoDelta, AutoMultivariateNormal
from pathlib import Path
import os
from jax import vmap
import jax.numpy as jnp
from jax.nn import softmax
import matplotlib.pyplot as plt
import numpy as np
from pandas.tseries.offsets import Day, BDay

# Antigen analysis package
from antigentools.io import (
    AntigenReader
)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import matplotlib.transforms as mtransforms
from evofr.plotting import *

font = {'family' : 'sans-serif',
        'serif': 'Helvetica Neue',
        'weight' : 'light',
        'size'   : 28}

matplotlib.rc('font', **font)

# Mute warnings
import warnings
warnings.filterwarnings('ignore')

/home/zthornto/miniforge3/envs/antigen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import yaml

with open("../config/config.yaml", 'r') as config:
    config = yaml.safe_load(config)
    
dates = config["main"]["estimation_dates"]
locations = config["main"]["locations"]
config["main"]["models"]

['MLR', 'FGA', 'GARW', 'naive']

## Helper functions

In [3]:
# Helper for MLR forecasting
def forecast_frequencies(samples, mlr, forecast_L=30):
    """
    Use posterior beta to forecast posterior frequenicies.

    Parameters
    ----------
    samples : dict
        Samples from the MCMC run.
    mlr : evofr.models.MultinomialLogisticRegression
        MLR model used for forecasting.
    forecast_L : int
        Number of days to forecast.
    
    Returns
    -------
    jnp.array
        Forecasted frequencies.
    """
    
    # Making feature matrix for forecasting
    last_T = samples["freq"].shape[1]
    X = mlr.make_ols_feature(start=last_T, stop=last_T + forecast_L)
    
    # Posterior beta
    beta = jnp.array(samples["beta"])
    
    # Matrix multiplication by sample
    dot_by_sample = vmap(jnp.dot, in_axes=(None, 0), out_axes=0)
    logits = dot_by_sample(X, beta) # Logit frequencies by variant
    return softmax(logits, axis=-1)

In [4]:
def save_freq(samples, variant_data, ps, forecast_date, name, filepath, num_nowcast_days = None):
    # Get nowcast frequencies
    freq_now = pd.DataFrame(ef.get_freq(samples, variant_data, ps, name=name, forecast = False))
    if num_nowcast_days:
        nowcast_dates = variant_data.dates[-num_nowcast_days:]
    else:
        nowcast_dates = variant_data.dates    
    freq_now = freq_now[freq_now['date'].isin(nowcast_dates)]
    
    # Get forecasted frequencies
    freq_fr = pd.DataFrame(ef.get_freq(samples, variant_data, ps, name=name, forecast = True))
    
    # Merge and export file
    freq_merged = pd.concat([freq_now, freq_fr])
    freq_merged = freq_merged.rename(columns = {'median_freq':'median_freq_nowcast'}, inplace = False)
    freq_merged.to_csv(f'{filepath}/freq_full_{forecast_date}.tsv', sep="\t", index = False)
    return None

In [5]:
def get_fixed_growth_advantage(samples, variant_data, ps, name,date, filepath, ref_variant="other", model_name=None):
    growth_adv = pd.DataFrame(ef.posterior.get_growth_advantage(samples, variant_data, ps, name=name, rel_to=ref_variant))
    growth_adv['date'] = date
    if model_name is not None:
        growth_adv['model'] = model_name
    growth_adv.to_csv(f'{filepath}/full_growth_advantages_{date}.tsv', sep="\t", index = False, header=True)
    return None

In [6]:
def get_time_varying_Rt(samples, variant_data, ps, name, date, filepath, model_name=None):
    """ Get time-varying Rt estimates from posterior and dump to file.

    Parameters
    ----------
    samples : dict
        Samples from the MCMC run.
    variant_data : evofr.data.VariantData
        Variant data object.
    ps : list
        List of quantiles to compute.
    name : str
        Name of the deme/location to consider.
    date : str
        Date of the analysis.
    site : str
        The summary statistic to compute.
    filepath : str
        Path to dump the file.
    """
    rt = pd.DataFrame(ef.posterior.get_site_by_variant(samples, variant_data, ps, name=name, site='R', forecast=False))
    rt_forecast = pd.DataFrame(ef.posterior.get_site_by_variant(samples, variant_data, ps, name=name, site='R', forecast=True))
    rt = pd.concat([rt, rt_forecast])
    if model_name is not None:
        rt['model'] = model_name
    rt.to_csv(f'{filepath}/full_rt_{date}.tsv', sep="\t", index = False, header = True)
    return None

## Naive model specification

Adding `forecast_L` argument to naive forecast -- default from Eslam's paper is set to 1.
This could be correct, -- I also haven't gotten any hindcast errors either

In [7]:
def naive_forecast(seq_count_date, pivot, n_days_to_average=7, period=30):
    """
    Naive forecast of the frequency of a variant.

    Parameters
    ----------
    seq_count_date : pd.DataFrame
        Sequence count data.
    pivot : str
        Pivot/forecasting date
    n_days_to_average : int
        Number of days to average counts over for forecasting.
    period : int
        Number of days to forecast.

    Returns
    -------
    pd.DataFrame
        Forecasted frequencies.
    """
    # Define dates for forecasting and nowcasting
    back_date = pd.to_datetime(pivot) - Day(period)
    forecast_dates = pd.to_datetime(pd.unique(pd.date_range(start=pivot, periods=period, freq='D'))).astype(str)
    nowcast_dates = pd.to_datetime(pd.unique(pd.date_range(start=back_date, periods=period, freq='D'))).astype(str)

    # Define prediction period for nowcasting and forecasting
    pred_dates = forecast_dates.union(nowcast_dates)

    # Compute frequency of variants for the demes using seq_count_data
    seq_count_date['total_seq'] = seq_count_date.groupby(['date', 'country'])['sequences'].transform('sum')
    seq_count_date['freq'] = seq_count_date['sequences'] / seq_count_date['total_seq']

    # Add prediction dates to date column for each country and variant
    sc_s = []
    for d in pred_dates:
        recent_dates = pd.Series(pd.to_datetime(seq_count_date[seq_count_date.date < d].date).unique()).nlargest(n_days_to_average).astype(str)

        # Get mean frequency of variants for the last 7 days
        seq_count_mean = seq_count_date[seq_count_date.date.isin(recent_dates)].groupby(["variant", "country"])["freq"].mean().reset_index()

        sc_ = seq_count_mean.copy()
        
        # Adding dates column
        sc_["date"] = d
        sc_s.append(sc_)

    sc = pd.concat(sc_s).sort_values(by=["country", "variant", "date"])
    
    # Adding nowcast and forecast columns
    sc['median_freq_nowcast'] = sc['freq']
    sc['median_freq_forecast'] = sc['freq']
    
    # Matching dates for nowcast and forecast
    sc.loc[sc.date.isin(forecast_dates),'median_freq_nowcast'] = np.nan
    sc.loc[sc.date.isin(nowcast_dates),'median_freq_forecast'] = np.nan
    return sc.reset_index(drop=True)

In [8]:
def get_reference_variant(variants_df: pd.DataFrame, window_variants: list) -> str:
    """
    Get the reference variant for a given time window.
    Reference variant is the variant that appears last in the window.

    Parameters
    ----------
    variants_df : pd.DataFrame
        Dataframe containing the variants.
    window_variants : list
        List of variants to consider for the reference variant.

    Returns
    -------
    str
        Reference variant.
    """
    # Subset the variants dataframe to only include the variants in the window
    vars_in_window = variants_df[variants_df.variant.isin(window_variants)]
    # Get the reference variant with the max value for birth
    return str(vars_in_window[vars_in_window.birth == vars_in_window.birth.max()].variant.values[0])

## Load and prep data

In [9]:
build = "flu-simulated-150k-samples-antigenic-clusters"

In [10]:
data_path = f'../data/{build}/time-stamped/*'
timepoints = glob.glob(data_path)
timepoints.sort()
# Remove truth folder
timepoints = [x for x in timepoints if 'truth' not in x]
# Get dates from timepoints paths
dates = [os.path.basename(x).split('_')[0] for x in timepoints]

In [11]:
# Load centroids
centroid_path = f'../data/{build}/clades/variant_centroids.csv'
centroids_df = pd.read_csv(centroid_path)
centroids_df.head()

,variant,ag1,ag2,birth,death
0,15,17.249568,2.194650,-0.0781,1.2527
1,24,17.338179,-2.811948,-0.0448,1.9359
2,27,19.793212,3.125662,0.2421,2.3052
3,34,17.897379,4.597683,0.3800,2.9919
4,2,18.807091,5.757923,0.5614,5.2014


### Model fitting parameters

In [12]:
seed_L = 14
forecast_L = 180
inference_method = ef.InferFullRank(iters=50_000, lr=0.01, num_samples=500)
ps = [0.95, 0.8, 0.5] # Levels of confidence to generate quantiles for
alphas = [0.2, 0.4, 0.6] # Transparency values for each quantile in ps

### Define forecast and nowcast dates

In [13]:
pivot = '2020-04-01'
period = 180
back_date = pd.to_datetime(pivot) - Day(forecast_L)
forecast_dates = pd.to_datetime(pd.unique(pd.date_range(start=pivot, periods=period, freq='D'))).astype(str)
nowcast_dates = pd.to_datetime(pd.unique(pd.date_range(start=back_date, periods=period, freq='D'))).astype(str)

## Model fitting

### Naive model

In [14]:
for pivot in dates:
    seq_count_date = pd.read_csv(f"../data/{build}/time-stamped/{pivot}/seq_counts.tsv", sep="\t")
    naive_pred = naive_forecast(seq_count_date, pivot, period=period)
    
    # Create files for estimates for each country and pivot date
    for location in locations:
        filepath = f'../results/{build}/estimates/naive/{location}/'
        if not os.path.exists(filepath):
            os.makedirs(filepath)
        naive_pred[naive_pred.country == location].to_csv(filepath + f"freq_full_{pivot}.tsv", sep="\t", index = False)

### MLR, FGA, GARW

Make sure we have loaded/defined `centroids_df` before this loop -- it should have the columns `birth` and `death`.

In [15]:
fail_counter = 0
ref_variant_dict = {'timepoint':[], 'country':[], 'ref_variant':[]}
for timepoint, date in zip(timepoints, dates):
    # Read in data
    seqs_df = pd.read_csv(timepoint + '/seq_counts.tsv', sep="\t")
    cases_df = pd.read_csv(timepoint + '/case_counts.tsv', sep="\t")
    
    for country in seqs_df.country.unique():
        model_dict = {}
        # Get country-specific data
        country_seqs_df = seqs_df[seqs_df.country == country]
        country_cases_df = cases_df[cases_df.country == country]
        if (len(country_seqs_df) == 0) or (len(country_cases_df) == 0) or (country_seqs_df.variant.nunique() == 1):
                print(f'No data for {date}: {country} deme')
                fail_counter += 1
                continue
        
        # Get variant names
        v_names = country_seqs_df.variant.unique().tolist()

        # Get reference variant
        ref_variant = get_reference_variant(centroids_df, v_names)
        ref_variant_dict['timepoint'].append(date)
        ref_variant_dict['country'].append(country)
        ref_variant_dict['ref_variant'].append(ref_variant)

        # Create case-frequency data set
        case_variant_data = ef.CaseFrequencyData(raw_cases=country_cases_df, raw_seq=country_seqs_df, var_names=v_names, pivot=ref_variant)
        variant_data = ef.VariantFrequencies(raw_seq=country_seqs_df, var_names=v_names, pivot=ref_variant)

        # Create delay padding embeddings for each variant
        gen = ef.pad_delays(
            [
                ef.discretise_gamma(mn=3.0, std=1.2) for _ in v_names
            ]
        )

        delays = ef.pad_delays([ef.discretise_lognorm(mn=3.1, std=1.0)])

        basis_fn = ef.Spline(order=4, k=10)

        # Define MLR model
        model_dict["MLR"] = ef.MultinomialLogisticRegression(tau=3.0)

        # Define growth advantage models
        model_dict["FGA"] = ef.RenewalModel(gen, delays, seed_L=seed_L, forecast_L=forecast_L, 
                                        RLik=ef.FixedGA(0.1), # Likelihood on effective reproduction number
                                        CLik=ef.ZINegBinomCases(0.05), # Case counts likelihood
                                        SLik = ef.DirMultinomialSeq(100), # Sequence counts likelihood
                                        v_names=v_names,
                                        basis_fn=basis_fn)
        model_dict["GARW"] = ef.RenewalModel(gen, delays, seed_L=seed_L, forecast_L=forecast_L,
                                    RLik=ef.GARW(0.1,0.01, prior_family='Normal'), # Likelihood on effective reproduction number (GARW depend on R and gen time
                                    CLik=ef.ZINegBinomCases(0.05), # Case counts likelihood
                                    SLik = ef.DirMultinomialSeq(100), # Sequence counts likelihood
                                    v_names=v_names,
                                    basis_fn=basis_fn)

        # Fit models
        for model in model_dict.keys():
            filepath = f'../results/{build}/estimates/{model}/{country}'
            if not os.path.exists(filepath):
                os.makedirs(filepath)
            print(f'{date}: Fitting {model} model for {country} deme....')
            
            if model == "MLR":
                try:
                    model_posterior = inference_method.fit(model_dict[model], variant_data)
                    model_posterior.samples['freq_forecast'] = forecast_frequencies(model_posterior.samples, model_dict[model], forecast_L=forecast_L)
                    save_freq(model_posterior.samples, variant_data, ps, date, country, filepath)
                    get_fixed_growth_advantage(model_posterior.samples, case_variant_data, ps, country, date, filepath, model_name=model)
                except:
                    fail_counter += 1
                    print(f'Failed to fit {model} model for {timepoint}...')
            else:
                try:
                    model_posterior = inference_method.fit(model_dict[model], case_variant_data)
                    save_freq(model_posterior.samples, case_variant_data, ps, date, country, filepath)
                    get_time_varying_Rt(model_posterior.samples, case_variant_data, ps, country, date, filepath, model_name=model)
                except:
                    fail_counter += 1
                    print(f'Failed to fit {model} model for {date}: {country} deme')

2020-04-01: Fitting MLR model for tropics deme....
2020-04-01: Fitting FGA model for tropics deme....
2020-04-01: Fitting GARW model for tropics deme....
2020-04-01: Fitting MLR model for north deme....
2020-04-01: Fitting FGA model for north deme....
2020-04-01: Fitting GARW model for north deme....
2020-04-01: Fitting MLR model for south deme....
2020-04-01: Fitting FGA model for south deme....
2020-04-01: Fitting GARW model for south deme....
2020-10-01: Fitting MLR model for tropics deme....
2020-10-01: Fitting FGA model for tropics deme....
2020-10-01: Fitting GARW model for tropics deme....
2020-10-01: Fitting MLR model for north deme....
2020-10-01: Fitting FGA model for north deme....
2020-10-01: Fitting GARW model for north deme....
2020-10-01: Fitting MLR model for south deme....
2020-10-01: Fitting FGA model for south deme....
2020-10-01: Fitting GARW model for south deme....
2021-04-01: Fitting MLR model for north deme....
2021-04-01: Fitting FGA model for north deme....
20

In [16]:
print(f'Failed to fit {fail_counter} timepoints out of {len(timepoints) * len(locations)} combinations')

Failed to fit 14 timepoints out of 180 combinations


In [17]:
# Save reference variant dictionary
ref_variant_df = pd.DataFrame(ref_variant_dict)
ref_variant_df.to_csv(f'../data/{build}/clades/ref_variants.tsv', sep="\t", index=False)